# Vérification pas à pas de preprocess_events.py

Importe les **vraies fonctions** de `scripts/preprocess_events.py` (aucune logique dupliquée ici) et les applique étape par étape sur des données réelles, en affichant la sortie à chaque étape — pour vérifier visuellement que chaque transformation fait ce qu'elle est censée faire, avant de faire confiance aux tests unitaires seuls.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..") / "scripts"))

from preprocess_events import (
    build_text,
    is_complete,
    is_relevant,
    load_raw_events,
    normalize_city,
    parse_labeled_field,
    preprocess,
    structure_event,
)

raw_events = load_raw_events()
print("Événements bruts chargés :", len(raw_events))

## Étape 1 — `is_relevant` : exclusion des sources hors-sujet

In [ ]:
relevant_events = [e for e in raw_events if is_relevant(e)]
excluded_events = [e for e in raw_events if not is_relevant(e)]

print(f"Avant filtrage : {len(raw_events)}")
print(f"Après filtrage : {len(relevant_events)}")
print(f"Exclus         : {len(excluded_events)}")
print()
print("Exemples exclus :")
for e in excluded_events[:3]:
    print(" -", e.get("title_fr"), "| source:", e.get("originagenda_title"))
print()
print("Exemples conservés :")
for e in relevant_events[:3]:
    print(" -", e.get("title_fr"), "| source:", e.get("originagenda_title"))

## Étape 2 — `normalize_city` : correction de la casse, sur un exemple réel

Recherche dans les données réelles d'un événement dont la ville est tout-majuscule, pour vérifier la correction sur un cas concret plutôt qu'un exemple inventé.

In [ ]:
all_caps_example = next(
    (e for e in relevant_events if (e.get("location_city") or "").isupper()), None
)

if all_caps_example:
    before = all_caps_example["location_city"]
    after = normalize_city(before)
    print(f"Avant : {before!r}")
    print(f"Après : {after!r}")
else:
    print("Aucun exemple tout-majuscule trouvé dans ce jeu de données.")

## Étape 3 — `parse_labeled_field` : décodage du statut, sur un événement annulé réel

In [ ]:
cancelled_example = next(
    (e for e in relevant_events if "Annul" in (e.get("status") or "")), None
)

if cancelled_example:
    print("Titre :", cancelled_example.get("title_fr"))
    print("status brut   :", cancelled_example["status"])
    print("status décodé :", parse_labeled_field(cancelled_example["status"]))
else:
    print("Aucun événement annulé trouvé dans ce jeu de données.")

## Étape 4 — `build_text` puis `structure_event` : avant / après complet, sur les mêmes exemples

In [ ]:
example = cancelled_example or relevant_events[0]

print("--- Champs bruts pertinents ---")
for field in ["title_fr", "description_fr", "status", "conditions_fr", "keywords_fr"]:
    print(f"{field}: {example.get(field)!r}")

print()
print("--- text vectorisé construit par build_text() ---")
print(build_text(example))

print()
print("--- Événement structuré complet (structure_event) ---")
structured_example = structure_event(example)
for key, value in structured_example.items():
    if key != "text":
        print(f"{key}: {value!r}")

## Étape 5 — `is_complete` : vérification sur un exemple valide et un exemple réellement rejeté

Recherche un événement réel dont le code postal ne commence pas par `13` (le cas Landerneau/Piolenc identifié dans `01_raw_data_exploration.ipynb`), pour vérifier que `is_complete()` le rejette bien — et confirme que l'exemple valide de l'étape 4 est accepté.

In [ ]:
mis_geocoded_raw = next(
    (
        e for e in relevant_events
        if (e.get("location_postalcode") or "").strip()
        and not e["location_postalcode"].startswith("13")
    ),
    None,
)

if mis_geocoded_raw:
    structured_bad = structure_event(mis_geocoded_raw)
    print("Titre          :", structured_bad["title"])
    print("location_city  :", structured_bad["location_city"])
    print("postalcode     :", structured_bad["location_postalcode"])
    print("is_complete()  :", is_complete(structured_bad))
else:
    print("Aucun événement mal géocodé trouvé dans ce jeu de données.")

print()
print("Exemple valide (étape 4) -> is_complete() :", is_complete(structured_example))

## Étape 6 — `preprocess()` : le pipeline complet, de bout en bout

`preprocess()` applique maintenant `is_relevant` puis `structure_event` puis `is_complete` — son propre message affiche le nombre d'événements à chaque étape du filtrage.

In [ ]:
processed_events = preprocess()
print("Événements structurés en sortie :", len(processed_events))
print("Champs par événement structuré  :", len(processed_events[0].keys()))

### Pourquoi exactement 8 exclusions (4244 → 4236) ?

L'étape 5 n'avait confirmé que 3 cas via le code postal (Landerneau x2, Piolenc x1), repérés indirectement dans `01_raw_data_exploration.ipynb` via les incohérences `location_insee`/`location_city`. Cette méthode ne pouvait trouver que les codes INSEE associés à **plusieurs** noms de ville différents — un événement avec un mauvais code postal mais un code INSEE utilisé nulle part ailleurs n'aurait jamais été détecté par cette méthode indirecte. `is_complete()`, lui, vérifie **directement et systématiquement** chaque événement, sans dépendre de ce genre de coïncidence. Décompte précis ci-dessous, par raison.

In [ ]:
relevant_structured = [structure_event(e) for e in relevant_events]

fails_text = [e for e in relevant_structured if not e["text"]]
fails_date = [e for e in relevant_structured if not e["date_start"]]
fails_uid = [e for e in relevant_structured if not e["uid"]]
fails_postalcode = [
    e for e in relevant_structured
    if e["location_postalcode"] and not e["location_postalcode"].startswith("13")
]
fails_in_person_location = [
    e for e in relevant_structured
    if e["attendance_mode"] != "En ligne"
    and not (e["location_name"] or e["location_address"] or e["location_city"])
]
fails_online_link = [
    e for e in relevant_structured
    if e["attendance_mode"] == "En ligne"
    and not (e["online_access_link"] or e["registration_link"])
]

print("Texte vide                          :", len(fails_text))
print("Date de début manquante             :", len(fails_date))
print("uid manquant                        :", len(fails_uid))
print("Code postal hors 13                 :", len(fails_postalcode))
print("Présentiel sans localisation         :", len(fails_in_person_location))
print("En ligne sans lien d'accès           :", len(fails_online_link))

excluded_uids = {
    e["uid"] for e in
    fails_text + fails_date + fails_uid + fails_postalcode + fails_in_person_location + fails_online_link
}
print()
print(f"Total événements uniques exclus : {len(excluded_uids)} / {len(relevant_structured)}")

print("\nDétail des événements exclus par code postal :")
for e in fails_postalcode:
    print(" -", e["title"], "| postalcode:", e["location_postalcode"], "| city:", e["location_city"])

## Étape 7 — Valeurs manquantes sur la sortie finale (24 champs)

Reprend l'audit de `01_raw_data_exploration.ipynb`, appliqué cette fois à la sortie réelle du pipeline complet (après `is_relevant` **et** `is_complete`) — pour comparer avant/après en toute traçabilité.

In [ ]:
def is_missing(value) -> bool:
    if value is None:
        return True
    if isinstance(value, str):
        return value.strip() == ""
    if isinstance(value, list):
        return len(value) == 0
    return False


total = len(processed_events)
fields = [f for f in processed_events[0].keys() if f not in ("age_min", "age_max")]

print(f"{'Champ':<22}{'Manquants':>12}{'Taux':>10}")
for field in fields:
    missing_count = sum(1 for e in processed_events if is_missing(e.get(field)))
    rate = missing_count / total * 100
    print(f"{field:<22}{missing_count:>12}{rate:>9.1f}%")

## Conclusion

Les règles primaires (texte vide, date/uid manquants, code postal hors Bouches-du-Rhône, présentiel sans localisation, en ligne sans lien d'accès) sont maintenant codées dans `is_complete()` et appliquées par `preprocess()` — plus une simulation, une vérification sur données réelles.

### Pourquoi les taux élevés de l'étape 7 ne posent pas de problème

Tous les champs avec un taux élevé de manquants (`conditions` 67,6%, `accessibility_labels` 87,4%, `keywords` 79,1%, `location_phone` 70,6%, `location_website` 69,8%, `location_links` 93,4%, `registration_link` 55,6%, `online_access_link` 96,8%) sont des champs qu'on a classés **secondaires** dès le début — ils enrichissent la réponse quand ils existent, mais leur absence n'empêche pas de répondre aux questions de base (quoi/quand/où). C'est exactement pour ça qu'on avait décidé de les garder malgré une faible fréquence de remplissage (choix délibéré de "garder le maximum de champs utiles" plutôt que de tout couper à cause d'un faible taux de remplissage).

`online_access_link` à 96,8% manquant, en particulier, est logique : la grande majorité des événements sont en présentiel (~4109 sur ~4236), donc ce champ est vide pour eux par construction, pas par erreur.

### Ce qui compte vraiment : les champs primaires

`uid`, `title`, `text`, `date_start`, `date_end`, `location_name`, `location_city`, `location_address`, `url` — **tous à 0,0% manquant**. Ce sont les seuls champs pour lesquels une absence aurait justifié une suppression, et `is_complete()` a déjà fait son travail : rien à ajouter.

### Un point mineur à noter, pas à corriger

`attendance_mode` a 48 manquants (1,1%). Ce n'est pas grave : la règle actuelle traite un mode manquant comme "présentiel par défaut" (`!= "En ligne"` est vrai si la valeur est `None`), donc ces 48 événements ne passent que s'ils ont bien une localisation — comportement prudent, pas une faille.

Prochaine étape : relancer `uv run python scripts/preprocess_events.py` pour régénérer `data/processed/events.json`, puis `uv run python scripts/build_index.py` pour reconstruire l'index FAISS avec ces corrections.